# Gabarito do Professor — Aula Integradora de Machine Learning

**Base:** `dados_educacionais_ml_5000.csv`  
**Equipe:** escreva aqui os nomes dos integrantes  
**Objetivo:** tratar os dados, responder perguntas de pesquisa, construir tabelas e gráficos e comparar modelos.

> Execute as células na ordem. Depois de cada resultado, registre uma interpretação em uma célula Markdown.

> **Material do professor:** contém resultados esperados, interpretações e alertas pedagógicos. Não distribuir antes da atividade.


## 1. Problema de pesquisa

Uma instituição de Educação Profissional quer compreender fatores relacionados à nota final, à aprovação e ao risco de evasão. A base é sintética e contém problemas intencionais de qualidade.

### Targets

- Classificação 1: `aprovado`
- Classificação 2 opcional: `risco_evasao`
- Regressão: `nota_final`

### Regra de segurança

O modelo deve apoiar uma análise pedagógica, nunca tomar sozinho uma decisão sobre um estudante.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except ImportError:
    display = print
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 50)
RANDOM_STATE = 42

## 2. Carregamento da base

Coloque o CSV na mesma pasta do notebook. No Google Colab, use o painel de arquivos para fazer o upload.

In [ ]:
ARQUIVO = "dados_educacionais_ml_5000.csv"

df_bruto = pd.read_csv(ARQUIVO)
df = df_bruto.copy()

print(f"Linhas: {df.shape[0]:,}")
print(f"Colunas: {df.shape[1]}")
display(df.head())

## 3. Auditoria inicial

Antes de limpar, responda:

1. Qual é a unidade de observação de cada linha?
2. Quais colunas são numéricas e quais são categóricas?
3. Quais colunas são identificadores?
4. Quais são features e quais são targets?

In [ ]:
df.info()

In [ ]:
resumo_qualidade_antes = pd.DataFrame({
    "tipo": df.dtypes.astype(str),
    "ausentes": df.isna().sum(),
    "ausentes_percentual": (df.isna().mean() * 100).round(2),
    "valores_unicos": df.nunique(dropna=False),
})

print("Duplicidades completas:", df.duplicated().sum())
display(resumo_qualidade_antes)

In [ ]:
colunas_categoricas = df.select_dtypes(include="object").columns

for coluna in colunas_categoricas:
    if coluna != "id_aluno":
        print(f"\n{coluna}")
        print(df[coluna].value_counts(dropna=False).head(15))

In [ ]:
limites_iniciais = df.select_dtypes(include="number").agg(
    ["min", "max", "mean", "median"]
).T.round(2)
display(limites_iniciais)

### Registro da auditoria

Complete esta seção antes de prosseguir:

| Problema encontrado | Coluna | Quantidade | Tratamento proposto | Justificativa |
|---|---|---:|---|---|
|  |  |  |  |  |

**Minha interpretação:** escreva aqui o que mais chamou a atenção na qualidade da base.

### Gabarito — auditoria inicial

Resultados esperados antes da limpeza:

| Problema | Evidência esperada |
|---|---:|
| Registros | 5.000 |
| Colunas | 21 |
| Duplicidades completas | 6 |
| `renda_familiar` ausente | 52 |
| `frequencia_percentual` ausente | 75 |
| `nota_anterior` ausente | 53 |
| `satisfacao_curso` ausente | 27 |
| IDs únicos | 4.994 |

Categorias inconsistentes esperadas:

- `desenv. sistemas`: 24 registros;
- `noturno` com diferença de capitalização/espaço: 20 registros;
- `Hibrido` sem acento: 17 registros;
- `SIM`: 15 registros;
- `nao`: 13 registros.

Também aparecem valores impossíveis, como idade 230, frequência 127%, nota anterior 11,8, horas de estudo negativas, 22 horas de sono, distância negativa, renda 250.000 e atividades entregues em 145%.

**Interpretação esperada:** a base não deve ser utilizada diretamente nos modelos. Os erros ilustram o princípio GIGO: mesmo um algoritmo correto pode aprender padrões distorcidos quando recebe dados inconsistentes.

## 4. Limpeza e padronização

O código abaixo fornece uma estratégia inicial. Leia cada operação, altere-a quando necessário e justifique suas decisões. Não existe tratamento automático perfeito.

In [ ]:
df = df_bruto.copy()

# 1. Remover duplicidades completas.
df = df.drop_duplicates().copy()

# 2. Remover espaços extras das colunas categóricas.
categoricas = df.select_dtypes(include="object").columns
for coluna in categoricas:
    df[coluna] = df[coluna].str.strip()

# 3. Padronizar categorias conhecidas.
df["curso"] = df["curso"].replace({
    "desenv. sistemas": "Desenvolvimento de Sistemas"
})
df["turno"] = df["turno"].str.title()
df["modalidade"] = df["modalidade"].replace({"Hibrido": "Híbrido"})

for coluna in ["internet_estavel", "trabalha", "recebeu_monitoria", "aprovado", "risco_evasao"]:
    df[coluna] = (
        df[coluna]
        .str.lower()
        .replace({"sim": "Sim", "não": "Não", "nao": "Não"})
    )

# 4. Valores fora de limites plausíveis tornam-se ausentes para posterior tratamento.
limites_validos = {
    "idade": (15, 80),
    "renda_familiar": (0, 50000),
    "horas_estudo_semana": (0, 60),
    "frequencia_percentual": (0, 100),
    "nota_anterior": (0, 10),
    "atividades_entregues_percentual": (0, 100),
    "horas_sono": (0, 16),
    "distancia_km": (0, 200),
    "satisfacao_curso": (1, 10),
    "nota_final": (0, 10),
}

for coluna, (minimo, maximo) in limites_validos.items():
    mascara_invalida = ~df[coluna].between(minimo, maximo) & df[coluna].notna()
    print(coluna, "valores impossíveis:", mascara_invalida.sum())
    df.loc[mascara_invalida, coluna] = np.nan

# 5. Imputação inicial pela mediana nas features numéricas.
# Compare esta decisão com dropna() e explique qual opção preserva melhor a atividade.
features_numericas = [
    "idade", "renda_familiar", "horas_estudo_semana", "frequencia_percentual",
    "nota_anterior", "acessos_ava_mes", "atividades_entregues_percentual",
    "horas_sono", "distancia_km", "satisfacao_curso"
]

for coluna in features_numericas:
    df[coluna] = df[coluna].fillna(df[coluna].median())

print("Dimensão antes:", df_bruto.shape)
print("Dimensão depois:", df.shape)

### Decisões de tratamento

Explique:

- por que a mediana foi escolhida;
- quantas linhas foram removidas;
- quais valores foram considerados impossíveis;
- quais valores apenas raros foram mantidos;
- uma limitação da estratégia adotada.

### Gabarito — decisões de limpeza

- A remoção das seis duplicidades reduz a base para **4.994 registros**.
- Textos devem ser tratados com `str.strip()`, uniformização de capitalização e substituições explícitas.
- Os oito valores impossíveis são transformados em `NaN` porque não há evidência para corrigir seus valores originais.
- A mediana é adequada como estratégia inicial porque sofre menos influência de extremos do que a média.
- Após a imputação apresentada, as features usadas nos modelos ficam sem valores ausentes.

**Limitação esperada:** imputar pela mediana reduz a variabilidade e pode ocultar diferenças entre grupos. Uma análise real deveria comparar métodos de imputação e documentar a origem do dado ausente.

In [ ]:
resumo_qualidade_depois = pd.DataFrame({
    "tipo": df.dtypes.astype(str),
    "ausentes": df.isna().sum(),
    "valores_unicos": df.nunique(dropna=False),
})

comparacao_qualidade = pd.DataFrame({
    "ausentes_antes": df_bruto.isna().sum(),
    "ausentes_depois": df.isna().sum(),
})

display(comparacao_qualidade)
print("Duplicidades depois:", df.duplicated().sum())

## 5. Outliers, percentis, IQR e BoxPlot

Um outlier não é automaticamente um erro. Use o IQR para sinalizar valores e depois investigue o contexto.

In [ ]:
def resumo_iqr(serie):
    q1 = serie.quantile(0.25)
    q3 = serie.quantile(0.75)
    iqr = q3 - q1
    limite_inferior = q1 - 1.5 * iqr
    limite_superior = q3 + 1.5 * iqr
    quantidade = ((serie < limite_inferior) | (serie > limite_superior)).sum()
    return pd.Series({
        "Q1": q1,
        "Mediana": serie.median(),
        "Q3": q3,
        "IQR": iqr,
        "Limite inferior": limite_inferior,
        "Limite superior": limite_superior,
        "Possíveis outliers": quantidade,
    })

display(df[features_numericas].apply(resumo_iqr).T.round(2))

In [ ]:
colunas_boxplot = [
    "renda_familiar", "horas_estudo_semana",
    "frequencia_percentual", "distancia_km"
]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for coluna, eixo in zip(colunas_boxplot, axes.ravel()):
    eixo.boxplot(df[coluna].dropna(), vert=False)
    eixo.set_title(coluna.replace("_", " ").title())
    eixo.set_xlabel("Valor")
plt.tight_layout()
plt.show()

**Pergunta:** quais valores extremos parecem plausíveis? Quais poderiam prejudicar um modelo baseado em distância?

### Gabarito — outliers

O IQR sinaliza, aproximadamente:

| Variável | Possíveis outliers |
|---|---:|
| Idade | 14 |
| Renda familiar | 204 |
| Horas de estudo | 15 |
| Frequência | 19 |
| Nota anterior | 15 |
| Acessos ao AVA | 19 |
| Atividades entregues | 18 |
| Horas de sono | 28 |
| Distância | 230 |
| Satisfação | 10 |

Os valores extremos restantes não são necessariamente erros. Uma renda de R$ 14.000 ou distância de 55 km pode ser rara, mas plausível. Para o KNN, renda e distância merecem atenção porque suas escalas podem dominar o cálculo das distâncias. A padronização dentro da pipeline reduz esse problema sem apagar observações plausíveis.

## 6. NumPy e representação dos dados

In [ ]:
array_numerico = df[features_numericas].to_numpy()

print("Tipo:", type(array_numerico))
print("Shape:", array_numerico.shape)
print("Dimensões:", array_numerico.ndim)
print("Quantidade de elementos:", array_numerico.size)

estatisticas_numpy = pd.DataFrame({
    "feature": features_numericas,
    "media": np.mean(array_numerico, axis=0),
    "mediana": np.median(array_numerico, axis=0),
    "minimo": np.min(array_numerico, axis=0),
    "maximo": np.max(array_numerico, axis=0),
}).round(2)
display(estatisticas_numpy)

## 7. Perguntas de pesquisa: tabelas

### Pergunta 1

Como a taxa de aprovação varia entre cursos, turnos e modalidades?

In [ ]:
df["aprovado_num"] = df["aprovado"].map({"Não": 0, "Sim": 1})
df["risco_evasao_num"] = df["risco_evasao"].map({"Não": 0, "Sim": 1})

aprovacao_curso = (
    df.groupby("curso", as_index=False)
      .agg(
          estudantes=("id_aluno", "count"),
          taxa_aprovacao=("aprovado_num", "mean"),
          nota_media=("nota_final", "mean"),
      )
      .sort_values("taxa_aprovacao", ascending=False)
)

aprovacao_curso["taxa_aprovacao"] = (aprovacao_curso["taxa_aprovacao"] * 100).round(1)
aprovacao_curso["nota_media"] = aprovacao_curso["nota_media"].round(2)
display(aprovacao_curso)

In [ ]:
tabela_curso_turno = pd.pivot_table(
    df,
    index="curso",
    columns="turno",
    values="aprovado_num",
    aggfunc="mean"
).mul(100).round(1)

display(tabela_curso_turno)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(aprovacao_curso["curso"], aprovacao_curso["taxa_aprovacao"], color="#2563EB")
ax.set_title("Taxa de aprovação por curso")
ax.set_xlabel("Curso")
ax.set_ylabel("Aprovação (%)")
ax.set_ylim(0, 100)
ax.tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()

**Resposta da equipe:** descreva o principal padrão e cite pelo menos dois valores da tabela.

### Pergunta 2

Qual variável parece ter maior relação com a nota final?

### Gabarito — aprovação por curso e turno

Taxa de aprovação por curso:

| Curso | Estudantes | Aprovação | Nota média |
|---|---:|---:|---:|
| Administração | 1.212 | 62,3% | 6,62 |
| Automação Industrial | 1.218 | 61,1% | 6,60 |
| Desenvolvimento de Sistemas | 1.303 | 60,5% | 6,57 |
| Eletromecânica | 1.261 | 59,4% | 6,62 |

Na tabela curso × turno, Administração Noturno apresenta aproximadamente 63,6% de aprovação, enquanto Eletromecânica Matutino apresenta cerca de 59,0%. As diferenças são pequenas. Não há base para afirmar que um curso ou turno causa maior aprovação.

In [ ]:
variaveis_engajamento = [
    "nota_final", "frequencia_percentual", "horas_estudo_semana",
    "atividades_entregues_percentual", "acessos_ava_mes"
]

correlacoes = df[variaveis_engajamento].corr().round(3)
display(correlacoes)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(df["frequencia_percentual"], df["nota_final"], alpha=0.25, s=14)
axes[0].set_title("Frequência e nota final")
axes[0].set_xlabel("Frequência (%)")
axes[0].set_ylabel("Nota final")

axes[1].scatter(df["atividades_entregues_percentual"], df["nota_final"], alpha=0.25, s=14, color="#D97706")
axes[1].set_title("Atividades entregues e nota final")
axes[1].set_xlabel("Atividades entregues (%)")
axes[1].set_ylabel("Nota final")

plt.tight_layout()
plt.show()

**Resposta da equipe:** qual associação foi mais forte? Correlação prova causalidade? Explique.

### Pergunta 3

Há diferenças observáveis entre estudantes que receberam ou não monitoria?

### Gabarito — engajamento e nota final

Correlações aproximadas com `nota_final`:

| Variável | Correlação |
|---|---:|
| Atividades entregues | 0,442 |
| Horas de estudo | 0,350 |
| Frequência | 0,272 |
| Acessos ao AVA | 0,199 |

Entre as variáveis solicitadas, `atividades_entregues_percentual` apresenta a associação linear mais forte. Ainda assim, correlação não prova causalidade. Estudantes mais organizados podem estudar mais, entregar mais atividades e obter notas maiores por fatores não medidos.

In [ ]:
comparacao_monitoria = (
    df.groupby("recebeu_monitoria", as_index=False)
      .agg(
          estudantes=("id_aluno", "count"),
          nota_media=("nota_final", "mean"),
          aprovacao=("aprovado_num", "mean"),
          risco_evasao=("risco_evasao_num", "mean"),
      )
)
comparacao_monitoria[["aprovacao", "risco_evasao"]] *= 100
display(comparacao_monitoria.round(2))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].bar(comparacao_monitoria["recebeu_monitoria"], comparacao_monitoria["nota_media"], color=["#64748B", "#2563EB"])
axes[0].set_title("Nota média por participação em monitoria")
axes[0].set_xlabel("Recebeu monitoria")
axes[0].set_ylabel("Nota média")

axes[1].bar(comparacao_monitoria["recebeu_monitoria"], comparacao_monitoria["risco_evasao"], color=["#64748B", "#DC2626"])
axes[1].set_title("Risco de evasão por participação em monitoria")
axes[1].set_xlabel("Recebeu monitoria")
axes[1].set_ylabel("Risco de evasão (%)")
plt.tight_layout()
plt.show()

**Resposta da equipe:** descreva as diferenças. Por que esta comparação não demonstra que a monitoria causou o resultado?

### Pergunta 4

Quais perfis concentram maior risco de evasão?

### Gabarito — monitoria

| Recebeu monitoria | Estudantes | Nota média | Aprovação | Risco de evasão |
|---|---:|---:|---:|---:|
| Não | 3.605 | 6,50 | 57,23% | 20,17% |
| Sim | 1.389 | 6,87 | 70,05% | 13,97% |

O grupo com monitoria apresenta melhor média, maior aprovação e menor risco. A comparação é observacional: quem procura monitoria pode ter maior motivação, apoio docente ou outras características. Portanto, os dados mostram associação, não efeito causal comprovado.

In [ ]:
dimensoes = ["curso", "turno", "internet_estavel", "trabalha", "recebeu_monitoria"]

for dimensao in dimensoes:
    tabela = (
        df.groupby(dimensao, as_index=False)["risco_evasao_num"]
          .mean()
          .sort_values("risco_evasao_num", ascending=False)
    )
    tabela["risco_evasao_percentual"] = (tabela.pop("risco_evasao_num") * 100).round(1)
    print(f"Risco por {dimensao}")
    display(tabela)

**Desafio:** crie um gráfico para uma das tabelas anteriores e investigue também satisfação, frequência ou distância usando faixas definidas com `pd.cut()`.

### Gabarito — perfis com maior risco de evasão

Principais resultados esperados:

- internet instável: **26,3%**, contra 16,7% entre quem possui internet estável;
- trabalha: **20,8%**, contra 13,8% entre quem não trabalha;
- não recebeu monitoria: **20,2%**, contra 14,0% entre quem recebeu;
- turno noturno: **19,0%**, contra 17,8% no matutino;
- Administração: aproximadamente 19,0%, maior entre os cursos, mas muito próxima dos demais.

A conclusão deve priorizar diferenças relevantes e evitar transformar perfis em rótulos individuais. O indicador pode orientar oferta de apoio, nunca punição.

## 8. Classificação — target `aprovado`

Não use `nota_final`, `risco_evasao` ou identificadores como features. `nota_final` determina diretamente a aprovação e causaria vazamento de dados.

In [ ]:
features_modelo = [
    "idade", "curso", "turno", "modalidade", "renda_familiar",
    "horas_estudo_semana", "frequencia_percentual", "nota_anterior",
    "acessos_ava_mes", "atividades_entregues_percentual", "horas_sono",
    "distancia_km", "internet_estavel", "trabalha", "recebeu_monitoria",
    "satisfacao_curso"
]

X = pd.get_dummies(df[features_modelo], drop_first=False, dtype=int)
y = df["aprovado_num"]

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Treino:", X_treino.shape)
print("Teste:", X_teste.shape)
print("Proporção de aprovação no treino:", y_treino.mean().round(3))
print("Proporção de aprovação no teste:", y_teste.mean().round(3))

### 8.1 Árvore de Decisão

In [ ]:
arvore = DecisionTreeClassifier(max_depth=5, random_state=RANDOM_STATE)
arvore.fit(X_treino, y_treino)
previsao_arvore = arvore.predict(X_teste)

acuracia_arvore = accuracy_score(y_teste, previsao_arvore)
print(f"Acurácia da Árvore: {acuracia_arvore:.3f}")
print(classification_report(y_teste, previsao_arvore, target_names=["Não", "Sim"]))

ConfusionMatrixDisplay.from_predictions(
    y_teste, previsao_arvore,
    display_labels=["Não", "Sim"],
    cmap="Blues"
)
plt.title("Matriz de confusão — Árvore de Decisão")
plt.show()

### 8.2 KNN com padronização e pipeline

Teste diferentes valores de K usando a mesma divisão de treino e teste.

In [ ]:
resultados_k = []

for k in [3, 5, 7, 9]:
    pipeline_knn = Pipeline([
        ("padronizacao", StandardScaler()),
        ("modelo", KNeighborsClassifier(n_neighbors=k))
    ])
    pipeline_knn.fit(X_treino, y_treino)
    previsao = pipeline_knn.predict(X_teste)
    resultados_k.append({"k": k, "acuracia": accuracy_score(y_teste, previsao)})

resultado_knn = pd.DataFrame(resultados_k).sort_values("acuracia", ascending=False)
display(resultado_knn)

In [ ]:
melhor_k = int(resultado_knn.iloc[0]["k"])
knn = Pipeline([
    ("padronizacao", StandardScaler()),
    ("modelo", KNeighborsClassifier(n_neighbors=melhor_k))
])
knn.fit(X_treino, y_treino)
previsao_knn = knn.predict(X_teste)
acuracia_knn = accuracy_score(y_teste, previsao_knn)

print(f"Melhor K: {melhor_k}")
print(f"Acurácia do KNN: {acuracia_knn:.3f}")
print(classification_report(y_teste, previsao_knn, target_names=["Não", "Sim"]))

ConfusionMatrixDisplay.from_predictions(
    y_teste, previsao_knn,
    display_labels=["Não", "Sim"],
    cmap="Oranges"
)
plt.title("Matriz de confusão — KNN")
plt.show()

In [ ]:
comparacao_classificadores = pd.DataFrame({
    "modelo": ["Árvore de Decisão", f"KNN (K={melhor_k})"],
    "acuracia": [acuracia_arvore, acuracia_knn]
}).sort_values("acuracia", ascending=False)

display(comparacao_classificadores)

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(comparacao_classificadores["modelo"], comparacao_classificadores["acuracia"], color=["#2563EB", "#D97706"])
ax.set_title("Acurácia dos classificadores")
ax.set_ylabel("Acurácia")
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

### Resposta sobre classificação

1. Qual modelo apresentou maior acurácia?
2. Qual apresentou menos falsos positivos?
3. Qual apresentou menos falsos negativos?
4. Qual erro seria mais preocupante neste contexto?
5. Qual modelo a equipe escolheria e por quê?

### Gabarito — comparação dos classificadores

Resultados com `random_state=42`:

| Modelo | Acurácia |
|---|---:|
| Árvore de Decisão (`max_depth=5`) | 0,840 |
| KNN (`K=5`) | 0,725 |

Matriz de confusão da Árvore, no formato `[[VN, FP], [FN, VP]]`:

```text
[[294, 98],
 [ 62, 545]]
```

Matriz de confusão do KNN:

```text
[[202, 190],
 [ 85, 522]]
```

A Árvore apresentou maior acurácia e menos falsos positivos e falsos negativos. Nesta base, é a escolha mais coerente. Entretanto, a justificativa deve mencionar também interpretabilidade, estabilidade e custo dos erros.

As variáveis mais importantes na Árvore foram frequência, atividades entregues, nota anterior e horas de estudo. Essa importância pertence ao modelo e à base usados; não equivale a causalidade.

## Gabarito adicional — segunda target de classificação

O indicador `risco_evasao` permite mostrar por que a acurácia isolada pode enganar. A classe “Sim” é minoritária.

In [ ]:
y_risco = df["risco_evasao_num"]

X_treino_r, X_teste_r, y_treino_r, y_teste_r = train_test_split(
    X, y_risco,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y_risco
)

arvore_risco = DecisionTreeClassifier(max_depth=5, random_state=RANDOM_STATE)
arvore_risco.fit(X_treino_r, y_treino_r)
previsao_risco = arvore_risco.predict(X_teste_r)

print(f"Acurácia: {accuracy_score(y_teste_r, previsao_risco):.3f}")
print(classification_report(y_teste_r, previsao_risco, target_names=["Não", "Sim"]))
print(confusion_matrix(y_teste_r, previsao_risco))

ConfusionMatrixDisplay.from_predictions(
    y_teste_r, previsao_risco,
    display_labels=["Não", "Sim"],
    cmap="Purples"
)
plt.title("Matriz de confusão — risco de evasão")
plt.show()

### Interpretação esperada da segunda target

A Árvore alcança aproximadamente **81,1% de acurácia**, mas a matriz é:

```text
[[795, 20],
 [169, 15]]
```

Ela identifica apenas 15 dos 184 estudantes com risco no conjunto de teste. Portanto, a acurácia aparentemente alta ocorre principalmente porque a classe “Não” é majoritária. Para uma intervenção preventiva, os 169 falsos negativos são especialmente graves.

Essa análise prepara a turma para precisão, recall, F1-score, balanceamento de classes e escolha de limiar.

## 9. Regressão Linear — target `nota_final`

Use a Regressão Linear para estimar a nota final. Avalie o erro no contexto da escala de 0 a 10.

In [ ]:
features_regressao = [
    "idade", "renda_familiar", "horas_estudo_semana",
    "frequencia_percentual", "nota_anterior", "acessos_ava_mes",
    "atividades_entregues_percentual", "horas_sono", "distancia_km",
    "satisfacao_curso"
]

X_reg = df[features_regressao]
y_reg = df["nota_final"]

X_treino_reg, X_teste_reg, y_treino_reg, y_teste_reg = train_test_split(
    X_reg, y_reg,
    test_size=0.20,
    random_state=RANDOM_STATE
)

regressao = Pipeline([
    ("padronizacao", StandardScaler()),
    ("modelo", LinearRegression())
])
regressao.fit(X_treino_reg, y_treino_reg)
previsao_reg = regressao.predict(X_teste_reg)

mae = mean_absolute_error(y_teste_reg, previsao_reg)
rmse = np.sqrt(mean_squared_error(y_teste_reg, previsao_reg))
r2 = r2_score(y_teste_reg, previsao_reg)

metricas_regressao = pd.DataFrame({
    "metrica": ["MAE", "RMSE", "R²"],
    "valor": [mae, rmse, r2]
})
display(metricas_regressao.round(3))

In [ ]:
comparacao_previsoes = pd.DataFrame({
    "nota_real": y_teste_reg.values,
    "nota_prevista": previsao_reg,
})
comparacao_previsoes["erro_absoluto"] = (
    comparacao_previsoes["nota_real"] - comparacao_previsoes["nota_prevista"]
).abs()

display(comparacao_previsoes.head(10).round(2))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(y_teste_reg, previsao_reg, alpha=0.35, s=18)
limite_min = min(y_teste_reg.min(), previsao_reg.min())
limite_max = max(y_teste_reg.max(), previsao_reg.max())
ax.plot([limite_min, limite_max], [limite_min, limite_max], "--", color="#DC2626", label="Previsão perfeita")
ax.set_title("Nota real × nota prevista")
ax.set_xlabel("Nota real")
ax.set_ylabel("Nota prevista")
ax.legend()
plt.tight_layout()
plt.show()

### Resposta sobre regressão

- O que o MAE representa em pontos de nota?
- Por que o RMSE pode ser maior que o MAE?
- O que o R² informa?
- Analise três previsões individuais.
- Em quais situações o modelo errou mais?

### Gabarito — Regressão Linear

Resultados aproximados:

| Métrica | Resultado | Interpretação |
|---|---:|---|
| MAE | 0,624 | erro absoluto médio de cerca de 0,62 ponto |
| RMSE | 0,774 | penaliza mais fortemente erros maiores |
| R² | 0,471 | o modelo explica aproximadamente 47,1% da variação das notas |

O modelo apresenta capacidade moderada. Ele pode servir como exercício de previsão, mas não é suficiente para definir o desempenho individual. Deve-se observar dispersão, resíduos e casos com erros maiores.

Exemplos da amostra de teste incluem: nota 8,8 prevista como 7,77; nota 6,5 prevista como 6,88; nota 5,6 prevista como 5,59. Os resultados podem variar minimamente entre versões das bibliotecas.

## 10. Conexão com Deep Learning

Nesta atividade, as features foram organizadas em uma matriz bidimensional:

- linhas = estudantes;
- colunas = características numéricas após a preparação.

Essa matriz pode ser vista como um tensor 2D. Redes neurais também exigiriam dados numéricos, escala coerente, divisão treino/teste e prevenção de vazamento.

In [ ]:
tensor_entrada = X.to_numpy(dtype=float)
print("Shape do tensor de entrada:", tensor_entrada.shape)
print("Dimensões:", tensor_entrada.ndim)

### Reflexão sobre Deep Learning

1. O que representam as duas dimensões do tensor?
2. Por que as categorias foram transformadas em números?
3. O que mudaria se cada observação fosse uma imagem?
4. Por que uma rede neural não elimina a necessidade de limpar os dados?

Não é necessário implementar uma rede neural.

### Gabarito — conexão com Deep Learning

Depois da codificação das categorias, a matriz possui `shape` aproximado de **(4.994, 24)**:

- 4.994 linhas representam estudantes após retirar duplicidades;
- 24 colunas representam features numéricas e categorias codificadas;
- trata-se de um tensor 2D;
- uma imagem individual normalmente seria um tensor 3D: altura × largura × canais;
- um lote de imagens normalmente seria 4D.

Redes neurais também precisam de dados numéricos, tratamento de ausentes, escalas coerentes e separação entre treino e teste. Deep Learning não corrige automaticamente dados ruins.

## 11. Conclusão final da equipe

Escreva de três a cinco parágrafos contendo:

1. principais problemas de qualidade encontrados;
2. respostas às perguntas de pesquisa com números;
3. modelo de classificação escolhido;
4. desempenho da Regressão Linear;
5. limitações da análise;
6. cuidados éticos e necessidade de revisão humana.

### Pergunta final

Se um modelo apresentar alta acurácia, mas errar principalmente os estudantes que precisam de apoio, ele deve ser adotado? Justifique.

### Modelo de conclusão esperada

A auditoria identificou ausências em quatro variáveis, seis duplicidades completas, categorias escritas de maneiras diferentes e oito valores impossíveis. Após remover duplicidades, padronizar categorias, transformar valores impossíveis em ausentes e aplicar imputação pela mediana, a base ficou com 4.994 registros preparados para a atividade.

As atividades entregues apresentaram a maior correlação com a nota final entre as variáveis de engajamento analisadas, cerca de 0,442. O grupo que recebeu monitoria apresentou nota média de 6,87 e aprovação de 70,05%, enquanto o grupo sem monitoria teve média 6,50 e aprovação de 57,23%. Essas diferenças representam associações e não demonstram causalidade.

Para prever aprovação, a Árvore de Decisão alcançou aproximadamente 84,0% de acurácia, superando o KNN, com cerca de 72,5%. Na regressão da nota final, o MAE foi aproximadamente 0,62 ponto e o R² cerca de 0,47. Os resultados mostram utilidade pedagógica, mas também revelam erros e variabilidade não explicada.

O caso da target de risco de evasão mostra que acurácia alta pode ocultar desempenho ruim na classe minoritária. Assim, nenhum modelo deve decidir sozinho quais estudantes receberão apoio. O uso responsável exige revisão humana, monitoramento de erros, transparência e proteção dos dados.